In [8]:
# Install transformers if needed
#!pip install transformers datasets sentencepiece

# Imports
import pandas as pd
import numpy as np
from transformers import BertTokenizer, XLMRobertaTokenizer
from transformers import BertForSequenceClassification, XLMRobertaForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader
import torch

# File upload (required)
from google.colab import files
uploaded = files.upload()  # Upload your dataset .csv files here


Saving Basics_of_BERT_and_XLM.zip to Basics_of_BERT_and_XLM.zip


# Understanding BERT & XLM-RoBERTa

## BERT
- Bidirectional encoder-only transformer.
- Uses Masked Language Modeling (MLM): 15% of tokens masked.
- Understands context from both left and right.
- Pre-trained on English (BooksCorpus + Wikipedia).
- Strength: deep semantic understanding for English tasks.

## XLM-RoBERTa
- Multilingual encoder-only model (100+ languages).
- Same MLM objective but trained on much larger CommonCrawl-MT.
- Robust cross-lingual transfer.
- Handles code-switching and non-Latin scripts very well.

## Tokenization differences
- BERT → WordPiece tokenizer, uses `[CLS]`, `[SEP]`
- XLM-R → SentencePiece tokenizer, uses `<s>`, `</s>`

## Key takeaway
BERT = Strong English semantic model  
XLM-R = Powerful multilingual generalist


In [9]:
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tok = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

sample_sentence = "Transformers are powerful models for NLP tasks."

# BERT tokenization
bert_out = bert_tok.encode_plus(
    sample_sentence,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

# XLM-R tokenization
xlm_out = xlm_tok.encode_plus(
    sample_sentence,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print("BERT input_ids:", bert_out["input_ids"])
print("BERT attention_mask:", bert_out["attention_mask"])
print("Decoded BERT:", bert_tok.decode(bert_out["input_ids"][0]))

print("\nXLM-R input_ids:", xlm_out["input_ids"])
print("XLM-R attention_mask:", xlm_out["attention_mask"])
print("Decoded XLM-R:", xlm_tok.decode(xlm_out["input_ids"][0]))


BERT input_ids: tensor([[  101, 19081,  2024,  3928,  4275,  2005, 17953,  2361,  8518,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]])
BERT attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])
Decoded BERT: [CLS] transformers are powerful models for nlp tasks. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]

XLM-R input_ids: tensor([[     0,  11062,  82772,      7,    621, 113138, 115774,    100,    541,
          37352,  66211,      7,      5,      2,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1]])
XLM-R attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [10]:
print("Special tokens BERT:", bert_tok.special_tokens_map)
print("Vocab size BERT:", bert_tok.vocab_size)

print("\nSpecial tokens XLM-R:", xlm_tok.special_tokens_map)
print("Vocab size XLM-R:", xlm_tok.vocab_size)


Special tokens BERT: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Vocab size BERT: 30522

Special tokens XLM-R: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab size XLM-R: 250002


In [11]:
import zipfile
import os
import pandas as pd

# get uploaded filename
zip_uploaded = list(uploaded.keys())[0]
print("Uploaded ZIP:", zip_uploaded)

# extract main zip
extract_dir = "/content/extracted_zip"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_uploaded, 'r') as z:
    z.extractall(extract_dir)

print("Top-level extracted:", os.listdir(extract_dir))

# find nested zip files (train.csv.zip, test.csv.zip)
nested_zips = []
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        if f.endswith(".zip"):
            nested_zips.append(os.path.join(root, f))

print("Nested ZIPs found:", nested_zips)

# extract nested zips
csv_paths = []
for zfile in nested_zips:
    with zipfile.ZipFile(zfile, 'r') as z:
        z.extractall(extract_dir)
        for name in z.namelist():
            if name.endswith(".csv"):
                csv_paths.append(os.path.join(extract_dir, name))

print("CSV files found:", csv_paths)

# choose train.csv automatically
train_csv = [p for p in csv_paths if "train" in p.lower()][0]
print("Using TRAIN CSV:", train_csv)

df = pd.read_csv(train_csv)
df.head(), df.shape


Uploaded ZIP: Basics_of_BERT_and_XLM.zip
Top-level extracted: ['test.csv', 'train.csv', 'Basics of BERT and XLM-RoBERTa - PyTorch']
Nested ZIPs found: ['/content/extracted_zip/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip', '/content/extracted_zip/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip']
CSV files found: ['/content/extracted_zip/train.csv', '/content/extracted_zip/test.csv']
Using TRAIN CSV: /content/extracted_zip/train.csv


(           id                                            premise  \
 0  5130fd2cb5  and these comments were considered in formulat...   
 1  5b72532a0b  These are issues that we wrestle with in pract...   
 2  3931fbe82a  Des petites choses comme celles-là font une di...   
 3  5622f0c60b  you know they can't really defend themselves l...   
 4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   
 
                                           hypothesis lang_abv language  label  
 0  The rules developed in the interim were put to...       en  English      0  
 1  Practice groups are not permitted to work on t...       en  English      2  
 2              J'essayais d'accomplir quelque chose.       fr   French      0  
 3  They can't defend themselves because of their ...       en  English      0  
 4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  ,
 (12120, 6))

In [14]:
print(df.columns)
df.head()


Index(['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label'], dtype='object')


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [15]:
# For NLI datasets, text input = (premise, hypothesis)
text_col_1 = "premise"
text_col_2 = "hypothesis"

label_col = "label"

print("Using text columns:", text_col_1, "and", text_col_2)
print("Using label column:", label_col)


Using text columns: premise and hypothesis
Using label column: label


In [17]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for train_idx, val_idx in kf.split(df, df[label_col]):
    folds.append((train_idx, val_idx))

print("Generated folds:", len(folds))


Generated folds: 5


In [18]:
class TextDataset(Dataset):
    def __init__(self, df, text1, text2, label, tokenizer, max_len=128):
        self.text1 = df[text1].tolist()
        self.text2 = df[text2].tolist()
        self.labels = df[label].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer.encode_plus(
            self.text1[idx],
            self.text2[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [19]:
train_idx, val_idx = folds[0]

train_df = df.iloc[train_idx]
val_df   = df.iloc[val_idx]

train_ds = TextDataset(train_df, text_col_1, text_col_2, label_col, bert_tok)
val_ds   = TextDataset(val_df, text_col_1, text_col_2, label_col, bert_tok)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)


In [20]:
num_labels = df[label_col].nunique()

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

model


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

EPOCHS = 1

for epoch in range(EPOCHS):
    print("Epoch", epoch+1)
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
    print("Loss:", loss.item())


Epoch 1
